# 03 Fine-tuning LoRA


## 환경


In [ ]:
from pathlib import Path
import os

from dotenv import load_dotenv

project_root = Path.cwd()
if not (project_root / ".env.example").exists():
  project_root = project_root.parent

load_dotenv(project_root / ".env")

base_model_id = os.getenv("HF_BASE_MODEL", "meta-llama/Meta-Llama-3-8B")
train_file = os.getenv("FINE_TUNE_TRAIN_FILE", "data/loyalty_qa_train.jsonl")
eval_file = os.getenv("FINE_TUNE_EVAL_FILE", "data/loyalty_qa_val.jsonl")
base_model_id, train_file, eval_file


## Dataset


In [ ]:
from datasets import load_dataset

train_dataset = load_dataset("json", data_files=train_file, split="train")
eval_dataset = load_dataset("json", data_files=eval_file, split="train")
train_dataset[:2]


## Model


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
  load_in_4bit=True,
  bnb_4bit_use_double_quant=True,
  bnb_4bit_quant_type="nf4",
  bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
  base_model_id,
  quantization_config=bnb_config,
  device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(
  base_model_id,
  padding_side="left",
  add_eos_token=True,
  add_bos_token=True,
)
tokenizer.pad_token = tokenizer.eos_token


## Tokenize


In [ ]:
def formatting_func(example):
  return f"### Question: {example['prompt']}\n### Answer: {example['response']}"

def tokenize_prompt(example):
  return tokenizer(formatting_func(example))

tokenized_train_dataset = train_dataset.map(tokenize_prompt)
tokenized_val_dataset = eval_dataset.map(tokenize_prompt)


## LoRA


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
  r=32,
  lora_alpha=64,
  target_modules=[
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
    "lm_head",
  ],
  bias="none",
  lora_dropout=0.05,
  task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## Train


In [ ]:
from datetime import datetime

import transformers

run_name = "llama3-lora-finetune"
output_dir = f"outputs/{run_name}"

trainer = transformers.Trainer(
  model=model,
  train_dataset=tokenized_train_dataset,
  eval_dataset=tokenized_val_dataset,
  args=transformers.TrainingArguments(
    output_dir=output_dir,
    warmup_steps=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,
    max_steps=200,
    learning_rate=2.5e-5,
    bf16=True,
    optim="paged_adamw_8bit",
    logging_steps=25,
    save_strategy="steps",
    save_steps=25,
    eval_strategy="steps",
    eval_steps=25,
    do_eval=True,
    report_to="none",
    run_name=f"{run_name}-{datetime.now().strftime('%Y-%m-%d-%H-%M')}",
  ),
  data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

model.config.use_cache = False
trainer.train()


## Inference


In [ ]:
def generate_text(user_prompt, max_new_tokens=100, repetition_penalty=1.15):
  model_input = tokenizer(user_prompt, return_tensors="pt").to("cuda")
  model.eval()
  with torch.no_grad():
    generated_output = model.generate(
      **model_input,
      max_new_tokens=max_new_tokens,
      repetition_penalty=repetition_penalty,
    )
  return tokenizer.decode(generated_output[0], skip_special_tokens=True)

generate_text("[MyElite Loyalty Program FAQ]:What is the maximum cashback I can earn?")
